In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , GridSearchCV , KFold , cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score , mean_squared_error , mean_absolute_error
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.base import BaseEstimator , TransformerMixin
import optuna
from lightgbm import LGBMRegressor
import mlflow
import dagshub
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate , plot_slice

C:\Users\LeoML\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dagshub.init(repo_owner="mridul0010" , repo_name="Delivery-Time-Analysis-Prediction" , mlflow=True)

Accessing as mridul0010

Initialized MLflow to track repo "mridul0010/Delivery-Time-Analysis-Prediction"

Repository mridul0010/Delivery-Time-Analysis-Prediction initialized!

In [ ]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow")

In [ ]:
mlflow.set_experiment("Hyperparameter Tuning - RF")

2026/07/02 00:31:19 INFO mlflow.tracking.fluent: Experiment with name 'Hyperparameter Tuning - RF' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/262c557fd773426192260f6ce2aaf745', creation_time=1782932481027, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1782932481027, lifecycle_stage='active', name='Hyperparameter Tuning - RF', tags={}, trace_location=None, workspace='default'>

In [ ]:
df = pd.read_csv('../data/Cleaned Delivery Dataset.csv')

In [ ]:
pd.set_option('display.max_columns' , None)

In [ ]:
df.drop(columns=['Delivery_Agent'] , inplace=True)

In [ ]:
df['Order_Datetime'] = pd.to_datetime(df['Order_Datetime'])
df['Pickup_Datetime'] = pd.to_datetime(df['Pickup_Datetime'])

In [ ]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.rating_bins = None

    def fit(self, X, y=None):
        X = X.copy()
        
        # Store bins for consistent transformation
        self.rating_bins = pd.qcut(
            X['Delivery_person_Ratings'],
            q=3,
            retbins=True,
            duplicates='drop'
        )[1]
        
        return self

    def transform(self, X):
        X = X.copy()

        # Datetime conversion
        X['Order_Datetime'] = pd.to_datetime(X['Order_Datetime'], errors='coerce')
        X['Pickup_Datetime'] = pd.to_datetime(X['Pickup_Datetime'], errors='coerce')

        # Rating group
        X["delivery_rating_group"] = pd.cut(
            X['Delivery_person_Ratings'],
            bins=self.rating_bins,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

        # Age group
        X["age_group"] = pd.cut(
            X['Delivery_person_Age'],
            bins=[14, 25, 35, 60],
            labels=['Young', 'Adult', 'Senior']
        )

        # Distance group
        X["distance_group"] = pd.cut(
            X['distance_km'],
            bins=[0, 5, 10, 25],
            labels=['Short Distance', 'Medium Distance', 'Long Distance']
        )

        # Time features
        X['Prep_Time(min)'] = (
            X['Pickup_Datetime'] - X['Order_Datetime']
        ).dt.total_seconds() / 60

        X['Order_hour'] = X['Order_Datetime'].dt.hour
        X['Order_day'] = X['Order_Datetime'].dt.day_name()

        X['isWeekend'] = X['Order_day'].isin(["Saturday", "Sunday"]).astype(int)

        X['Time_Of_Day'] = pd.cut(
            X['Order_hour'],
            bins=[0, 6, 12, 18, 24],
            labels=["Night", "Morning", "Afternoon", "Evening"],
            include_lowest=True
        )

        # Drop raw datetime
        X = X.drop(columns=['Order_Datetime', 'Pickup_Datetime'], errors='ignore')

        return X

    def get_feature_names_out(self , input_features = None):
        return input_features

In [ ]:
X = df.drop(columns=['Time_taken (min)'])
y = df['Time_taken (min)']

In [ ]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [ ]:
X

,City,Zone,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,distance_km,Order_Datetime,Pickup_Datetime,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City_Type
0,Dehradun,Zone17,36,4.2,30.327968,78.046106,30.397968,78.116106,10.28,2022-12-02 21:55:00,2022-12-02 22:10:00,Fog,Jam,Good,Snack,motorcycle,3,No,Metropolitian
1,Kochi,Zone16,21,4.7,10.003064,76.307589,10.043064,76.347589,6.24,2022-02-13 14:55:00,2022-02-13 15:05:00,Stormy,High,Average,Meal,motorcycle,1,No,Metropolitian
2,Pune,Zone13,23,4.7,18.562450,73.916619,18.652450,74.006619,13.79,2022-04-03 17:30:00,2022-04-03 17:40:00,Sandstorms,Medium,Average,Drinks,scooter,1,No,Metropolitian
3,Ludhiana,Zone15,34,4.3,30.899584,75.809346,30.919584,75.829346,2.93,2022-02-13 09:20:00,2022-02-13 09:30:00,Sandstorms,Low,poor,Buffet,motorcycle,0,No,Metropolitian
4,Kanpur,Zone14,24,4.7,26.463504,80.372929,26.593504,80.502929,19.40,2022-02-14 19:50:00,2022-02-14 20:05:00,Fog,Jam,Average,Snack,scooter,1,No,Metropolitian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39637,Bangalore,Zone16,28,4.9,13.029198,77.570997,13.059198,77.600997,4.66,2022-03-30 21:55:00,2022-03-30 22:05:00,Sandstorms,Jam,Average,Meal,scooter,1,No,Metropolitian
39638,Ranchi,Zone16,35,4.2,23.371292,85.327872,23.481292,85.437872,16.60,2022-08-03 21:45:00,2022-08-03 21:55:00,Windy,Jam,Good,Drinks,motorcycle,1,No,Metropolitian
39639,Chennai,Zone08,30,4.9,13.022394,80.242439,13.052394,80.272439,4.66,2022-11-03 23:50:00,2022-11-04 00:05:00,Cloudy,Low,Average,Drinks,scooter,0,No,Metropolitian
39640,Coimbatore,Zone11,20,4.7,11.001753,76.986241,11.041753,77.026241,6.23,2022-07-03 13:35:00,2022-07-03 13:40:00,Cloudy,High,poor,Snack,motorcycle,1,No,Metropolitian


In [ ]:
numerical = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 
       'multiple_deliveries', 'distance_km', 'Prep_Time(min)', 'Order_hour',
       'isWeekend']

ohe_col = ['City', 'Zone', 'Weather_conditions',
       'Type_of_order', 'Type_of_vehicle',
       'City_Type','Order_day', 'Time_Of_Day']

ordinal_col = ['Road_traffic_density' , 'Vehicle_condition' , 'Festival',
       'delivery_rating_group' , 'age_group' , 'distance_group']

In [ ]:
transformer = ColumnTransformer(
    transformers=[
        ("ohe" , OneHotEncoder(sparse_output=False , handle_unknown='ignore') , ohe_col),
        ('oe' , OrdinalEncoder(categories=[
            ['Low', 'Medium', 'High', 'Jam'],                         # Road_traffic_density
            ['poor', 'Average', 'Good' , 'Excellent'],                # Vehicle_condition
            ['No', 'Yes'],                                            # Festival
            ['Low', 'Medium', 'High'],                                # delivery_rating_group
            ['Young', 'Adult', 'Senior'],                             # age_group
            ['Short Distance', 'Medium Distance', 'Long Distance']    # distance_group
        ]) , ordinal_col),
        ('Scaling' , StandardScaler() , numerical)
    ],remainder='passthrough'
)

In [ ]:
pipeline = Pipeline(
    [
        ("FeatureEngineering" , FeatureEngineering()),
        ("ColumnTransformer" , transformer)
    ]
) 

In [ ]:
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)

In [ ]:
pt = PowerTransformer()

In [ ]:
def objective(trial):
    with mlflow.start_run(nested=True):
        params = {
            # Number of trees in the forest
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            
            # Maximum depth of the tree
            "max_depth": trial.suggest_int("max_depth", 2, 32),
            
            # Minimum number of samples required to split an internal node
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            
            # Minimum number of samples required to be at a leaf node
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            
            # Number of features to consider when looking for the best split
            "max_features": trial.suggest_float("max_features", 0.1, 1.0),
            
            # Whether bootstrap samples are used when building trees
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
            
            "random_state": 42,
            "n_jobs": -1,
        }

        # log params
        mlflow.log_params(params)


        rf = RandomForestRegressor(**params)
        model = TransformedTargetRegressor(regressor=rf , transformer=pt)

        model.fit(X_train_trans , y_train)

        cv_score = cross_val_score(
            model ,
            X_train_trans , 
            y_train , 
            cv=10 , 
            scoring="neg_mean_absolute_error",
            n_jobs=-1
        )

        mean_score = -(cv_score.mean())

        # log avg cross val error
        mlflow.log_metric("cross_val_error" , mean_score)

        return mean_score

In [ ]:
study = optuna.create_study(direction='minimize')

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective , n_trials=50 , n_jobs=-1 , show_progress_bar=True)

    # log the best params
    mlflow.log_params(study.best_params)

    # log best score
    mlflow.log_metric("best_score" , study.best_value)

    # training the lgbm on best param
    best_rf = RandomForestRegressor(**study.best_params)

    best_model = TransformedTargetRegressor(regressor=best_rf , transformer=pt)

    best_model.fit(X_train_trans , y_train)

    y_pred_train = best_model.predict(X_train_trans)
    y_pred_test = best_model.predict(X_test_trans)

    scores = cross_val_score(
        best_model,
        X_train_trans,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=5,n_jobs=-1
    )

    # logging metrics
    mlflow.log_metric("Training_error" ,mean_absolute_error(y_train ,y_pred_train))
    mlflow.log_metric("Test_error" ,mean_absolute_error(y_test ,y_pred_test))
    mlflow.log_metric("Training_r2" ,r2_score(y_train ,y_pred_train))
    mlflow.log_metric("Test_r2" ,r2_score(y_test ,y_pred_test))
    mlflow.log_metric("cross_val" , -scores.mean())

    # Generate the optuna plots
    fig_history = plot_optimization_history(study)
    fig_parallel = plot_parallel_coordinate(study)
    fig_importance = plot_param_importances(study)
    fig_slice = plot_slice(study)

    # Loginf plots
    mlflow.log_figure(fig_history, "optuna_plots/optimization_history.html")
    mlflow.log_figure(fig_importance, "optuna_plots/param_importances.html")
    mlflow.log_figure(fig_parallel, "optuna_plots/parallel_coordinate.html")
    mlflow.log_figure(fig_slice, "optuna_plots/plot_slice.html")
    
    # log the best model 
    mlflow.sklearn.log_model(best_rf , artifact_path="model_RF")

[I 2026-07-02 00:31:20,444] A new study created in memory with name: no-name-c9bda206-13d4-49ee-ab1e-c01c51a68d72
  0%|          | 0/50 [00:00<?, ?it/s]

🏃 View run resilient-shad-423 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/0/runs/8970ac84fb3640e795668685b67138c3
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/0


Best trial: 12. Best value: 4.06613:   2%|▏         | 1/50 [02:12<1:48:18, 132.61s/it]

[I 2026-07-02 00:33:33,757] Trial 12 finished with value: 4.066134063951305 and parameters: {'n_estimators': 111, 'max_depth': 6, 'min_samples_split': 9, 'min_samples_leaf': 16, 'max_features': 0.2511049179853957, 'bootstrap': True}. Best is trial 12 with value: 4.066134063951305.
🏃 View run peaceful-mare-906 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/0/runs/5eeb35af30ad405eb3343d873b311e95
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/0


Best trial: 1. Best value: 3.06115:   4%|▍         | 2/50 [02:17<46:12, 57.75s/it]    

...

[I 2026-07-02 01:04:09,679] Trial 48 finished with value: 3.047789164134227 and parameters: {'n_estimators': 275, 'max_depth': 16, 'min_samples_split': 18, 'min_samples_leaf': 4, 'max_features': 0.49057667375280534, 'bootstrap': False}. Best is trial 33 with value: 3.032367779461439.
🏃 View run intrigued-shark-391 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/0/runs/26a9ff26be8248d49b5b12b7cb43c0e7
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/0


Best trial: 33. Best value: 3.03237: 100%|██████████| 50/50 [33:09<00:00, 39.80s/it]


[I 2026-07-02 01:04:30,939] Trial 49 finished with value: 3.0478927903452697 and parameters: {'n_estimators': 272, 'max_depth': 16, 'min_samples_split': 18, 'min_samples_leaf': 4, 'max_features': 0.48561750394769165, 'bootstrap': False}. Best is trial 33 with value: 3.032367779461439.


2026/07/02 01:05:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/02 01:05:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/0/runs/9949d7127ba241fc8de57c04e09916ee
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/0


In [ ]:
study.best_value

3.032367779461439

In [ ]:
study.best_params

{'n_estimators': 299,
 'max_depth': 13,
 'min_samples_split': 20,
 'min_samples_leaf': 2,
 'max_features': 0.44603640869435546,
 'bootstrap': False}

In [ ]:
best_rf = RandomForestRegressor(**study.best_params)

model = TransformedTargetRegressor(regressor=best_rf , transformer=pt)

model.fit(X_train_trans , y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",RandomForestR...stimators=299)
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",PowerTransformer()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",299
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",13
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",20
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of

In [ ]:
y_pred = model.predict(X_test_trans)

r2_score(y_test , y_pred)

0.8335052876962259

In [ ]:
study.trials_dataframe().head()

,number,value,datetime_start,datetime_complete,duration,params_bootstrap,params_max_depth,params_max_features,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,3.072966,2026-07-02 00:31:21.152154,2026-07-02 00:36:31.759174,0 days 00:05:10.607020,False,18,0.575202,17,17,165,COMPLETE
1,1,3.061151,2026-07-02 00:31:21.153494,2026-07-02 00:33:39.105108,0 days 00:02:17.951614,True,14,0.547723,8,20,253,COMPLETE
2,2,3.076381,2026-07-02 00:31:21.154531,2026-07-02 00:33:56.550174,0 days 00:02:35.395643,False,28,0.592388,20,16,242,COMPLETE
3,3,3.072054,2026-07-02 00:31:21.155375,2026-07-02 00:38:47.102166,0 days 00:07:25.946791,True,25,0.756241,11,13,201,COMPLETE
4,4,3.071098,2026-07-02 00:31:21.156738,2026-07-02 00:36:46.851322,0 days 00:05:25.694584,True,30,0.293992,6,14,147,COMPLETE


In [ ]:
plot_optimization_history(study)

In [ ]:
plot_param_importances(study)

In [ ]:
plot_slice(study)

In [ ]:
plot_parallel_coordinate(study)